# Implementing a Custom GeoDataset

This notebook walks through how to implement your own geospatial dataset for use with DDR by subclassing `BaseGeoDataset`. The MERIT Hydro implementation is used as the concrete example throughout.

After reading this notebook you will know:
- What data your dataset needs to provide and in what format
- What each abstract method must do, with the shape and semantic contracts spelled out
- The subtle invariants (unit conversions, tensor transpositions) that are easy to get wrong silently
- How to plug your custom class into a training or routing run

---
## 1. How the DataLoader and routing engine interact with your dataset

Before writing any code it helps to see where your dataset sits in the overall data flow.

```
TRAINING MODE
─────────────

  __init__
    └─ _init_training()          ← you implement this
         sets: self.gage_ids,
               self.observations,
               self.gages_adjacency,
               self.obs_reader

  DataLoader  →  collate_fn()    ← provided by BaseGeoDataset
                   │  calls dates.calculate_time_period()
                   └─ _collate_gages(batch)  ← you implement this
                        │  calls _build_common_tensors()
                        └─ RoutingDataclass
                               │
                               ▼
                          dmc (MC routing engine)


INFERENCE MODE  (testing / routing)
────────────────────────────────────

  __init__
    └─ _init_inference()         ← you implement this
         sets: self.routing_dataclass
               (calls _build_common_tensors() internally)

  DataLoader  →  collate_fn()    ← provided by BaseGeoDataset
                   │  reads self.routing_dataclass directly,
                   │  slices dates for each chunk
                   └─ RoutingDataclass
                               │
                               ▼
                          dmc (MC routing engine)
```

The `BaseGeoDataset` provides the PyTorch `Dataset` scaffolding (`__len__`, `__getitem__`, `collate_fn`). You implement the six abstract methods that fill in the domain-specific data.

---
## 2. What data your dataset must provide

| Data | Format | Purpose |
|---|---|---|
| **Catchment attributes** | `xr.Dataset` with a spatial coord (e.g. `COMID`) | Input features to the KAN |
| **Flowpath attributes** | `gpd.GeoDataFrame` indexed by catchment ID | Physical channel geometry for MC routing |
| **CONUS adjacency** | zarr group (loaded by `read_zarr`) | Full directed flow network |
| **Gages adjacency** *(training)* | zarr group | Pre-built per-gage subgraphs |
| **Streamflow observations** *(training/testing)* | `xr.Dataset` from `IcechunkUSGSReader` | Training targets |

The `data_sources` section of your config points to all of these files. You do not need to change the config schema — the existing fields cover both MERIT and custom datasets.

### Flowpath columns your dataset must supply

| Column | Units | Required? | Notes |
|---|---|---|---|
| Reach length | **metres** | Yes | MERIT stores `lengthkm`; multiply by 1000 |
| Channel slope | dimensionless | Yes | |
| Muskingum *x* | 0–0.5 | Yes | Lynker reads per-reach; MERIT uses fixed 0.3 |
| Top width | metres | No | Set `torch.empty(0)` if using Leopold & Maddock geometry |
| Side slope *z* | H:V | No | Set `torch.empty(0)` if using Leopold & Maddock geometry |

---
## 3. Imports and the abstract interface

The cell below shows the imports your subclass will need and the abstract class signature.

In [ ]:
from pathlib import Path
from typing import Any

import geopandas as gpd
import numpy as np
import torch
import xarray as xr
from scipy import sparse

from ddr.geodatazoo.base_geodataset import BaseGeoDataset
from ddr.geodatazoo.dataclasses import Dates, RoutingDataclass
from ddr.io.builders import construct_network_matrix, create_hydrofabric_observations
from ddr.io.readers import (
    IcechunkUSGSReader,
    build_flow_scale_tensor,
    fill_nans,
    filter_headwater_gages,
    naninfmean,
    read_zarr,
)
from ddr.io.statistics import set_statistics
from ddr.validation.configs import Config
from ddr.validation.enums import Mode

---
## 4. Step-by-step: the six abstract methods

We walk through each method in the order it is called, using the real MERIT implementation.

### 4.1 `__init__` — constructor pattern

`BaseGeoDataset` does not define `__init__`, so yours is the only one. The pattern used by both built-in datasets is:

1. Store the config and build a `Dates` object.
2. Declare all instance attributes as `None` (mypy requires this).
3. Call `_load_attributes()` and compute normalisation statistics.
4. Load the flowpath geometry.
5. Load the CONUS adjacency zarr.
6. Delegate to `_init_training()` or `_init_inference()` based on `cfg.mode`.

Steps 1–5 run for every mode, so keep them cheap. The per-mode initialisation in step 6 is where expensive observation loading or graph traversal happens.

In [ ]:
# Constructor pattern — illustrative skeleton, not executable on its own

class MyDataset(BaseGeoDataset):
    def __init__(self, cfg: Config) -> None:
        self.cfg = cfg
        self.dates = Dates(**self.cfg.experiment.model_dump())

        # Declare optional attributes so type-checkers are happy
        self.gage_ids: np.ndarray | None = None
        self.routing_dataclass: RoutingDataclass | None = None
        self.observations: Any = None
        self.gages_adjacency: Any = None

        # Load attributes and compute statistics (used for normalisation)
        self.attribute_ds = self._load_attributes()
        self.attr_stats = set_statistics(self.cfg, self.attribute_ds)
        self.attributes_list = list(self.cfg.kan.input_var_names)

        # Pre-build mean/std tensors so normalisation is a single broadcast op
        self.means = torch.tensor(
            [self.attr_stats[attr].iloc[2] for attr in self.attributes_list],
            device=self.cfg.device, dtype=torch.float32,
        ).unsqueeze(1)  # shape (num_attrs, 1) for broadcasting over N catchments
        self.stds = torch.tensor(
            [self.attr_stats[attr].iloc[3] for attr in self.attributes_list],
            device=self.cfg.device, dtype=torch.float32,
        ).unsqueeze(1)

        # ... load flowpath geometry and adjacency zarr ...

        if cfg.mode == Mode.TRAINING:
            self._init_training()
        else:
            self._init_inference()

### 4.2 `_load_attributes` — load your attribute store

**Contract:** return an `xr.Dataset` whose data variables cover every name in `cfg.kan.input_var_names`, with a spatial coordinate that you can use to select individual catchments.

MERIT stores attributes in a NetCDF file indexed by `COMID`. Lynker uses an icechunk store indexed by `divide_id`. The format does not matter as long as the result is an `xr.Dataset`.

In [ ]:
# MERIT implementation
def _load_attributes(self) -> xr.Dataset:
    # xr.open_mfdataset handles both single files and glob patterns
    return xr.open_mfdataset(self.cfg.data_sources.attributes)

### 4.3 `_get_attributes` — fetch attributes for a batch of catchment IDs

**Output shape: `(num_attributes, N)` — attributes are rows, catchments are columns.**

This feels backwards compared to the usual ML convention of `(N, features)`, but it makes the per-attribute NaN-filling and normalisation arithmetic simpler (a single row operation rather than a column operation).

Two things to handle carefully:
- **Missing catchments**: some IDs in `catchment_ids` may not be present in your attribute store (edge segments, cross-boundary reaches). Pre-fill the output tensor with `np.nan` and write a validity mask — `fill_nans` will replace them with per-attribute means.
- **ID type**: MERIT COMIDs are integers; Lynker divide IDs are strings. Ensure your `id_to_index` lookup dict uses the same type as the incoming `catchment_ids`.

In [ ]:
# MERIT implementation
def _get_attributes(
    self,
    catchment_ids: np.ndarray,
    device: str | torch.device = "cpu",
    dtype: torch.dtype = torch.float32,
) -> torch.Tensor:
    valid_indices = []   # positions in the attribute store
    divide_idx_mask = [] # positions in the output tensor where data will be written

    for i, divide_id in enumerate(catchment_ids):
        comid = int(divide_id)  # MERIT IDs are integers
        if comid in self.id_to_index:
            valid_indices.append(self.id_to_index[comid])
            divide_idx_mask.append(i)
        # Missing IDs are silently skipped; their column stays NaN

    assert valid_indices, "No valid COMIDs found in this batch"

    # Pre-fill with NaN so fill_nans can substitute per-attribute means later
    output = torch.full(
        (len(self.attributes_list), len(catchment_ids)),
        np.nan, device=device, dtype=dtype,
    )  # shape: (num_attributes, N)

    _ds = self.attribute_ds[self.attributes_list].isel(COMID=valid_indices).compute()
    data_tensor = torch.from_numpy(
        _ds.to_array(dim="COMID").values  # also (num_attributes, N_valid)
    ).to(device=device, dtype=dtype)

    output[:, divide_idx_mask] = data_tensor
    return fill_nans(attr=output, row_means=self.means)

### 4.4 `_build_common_tensors` — the core tensor assembly step

This method is called by `_collate_gages`, `_build_routing_data_gages`, `_build_routing_data_target_catchments`, and `_build_routing_data_all_catchments`. All four paths need the same four outputs, so the logic is centralised here.

**Return values:**

| Variable | Shape | Notes |
|---|---|---|
| `adjacency_matrix` | `(N, N)` sparse CSR | On `cfg.device` |
| `spatial_attributes` | `(num_attributes, N)` | Raw, NaNs filled |
| `normalized_spatial_attributes` | `(N, num_attributes)` | Z-score normalised, **transposed** |
| `flowpath_tensors` | `dict[str, Tensor]` | Keys: `length`, `slope`, `x`, `top_width`, `side_slope` |

> **Why the transposition?** `spatial_attributes` is `(num_attrs, N)` because that layout makes per-attribute operations like `(attr - mean) / std` a single broadcast. But the KAN expects `(N, num_features)`. The transpose in `_build_common_tensors` is the only place this conversion happens — do not add it elsewhere.

> **`length` units:** the MC engine expects **metres**. MERIT stores reach length in kilometres (`lengthkm`), so you must multiply by 1000. Lynker already stores `Length_m` in metres.

> **Unused geometry tensors:** if your dataset uses Leopold & Maddock power-law geometry (MERIT), set `top_width` and `side_slope` to `torch.empty(0)`. The routing engine checks the tensor size to decide which geometry branch to use.

In [ ]:
# MERIT implementation
def _build_common_tensors(
    self,
    csr_matrix: sparse.csr_matrix,       # (N, N) scipy CSR for the subgraph
    catchment_ids: np.ndarray,            # (N,) IDs in compressed-index order
    flowpath_attr: gpd.GeoDataFrame,      # indexed by catchment ID
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, dict[str, torch.Tensor]]:

    # 1. Adjacency matrix — convert scipy CSR to a PyTorch sparse CSR tensor
    adjacency_matrix = torch.sparse_csr_tensor(
        crow_indices=csr_matrix.indptr,
        col_indices=csr_matrix.indices,
        values=csr_matrix.data,
        size=csr_matrix.shape,
        device=self.cfg.device,
        dtype=torch.float32,
    )  # shape (N, N)

    # 2. Raw attributes — shape (num_attributes, N)
    spatial_attributes = self._get_attributes(
        catchment_ids=catchment_ids,
        device=self.cfg.device,
        dtype=torch.float32,
    )

    # Fill any remaining per-row NaNs with the batch mean
    for r in range(spatial_attributes.shape[0]):
        row_means = torch.nanmean(spatial_attributes[r])
        nan_mask = torch.isnan(spatial_attributes[r])
        spatial_attributes[r, nan_mask] = row_means

    # 3. Z-score normalise then TRANSPOSE from (num_attrs, N) → (N, num_attrs)
    #    self.means and self.stds are both (num_attrs, 1), so the broadcast
    #    operates over the N dimension automatically.
    normalized_spatial_attributes = (spatial_attributes - self.means) / self.stds
    normalized_spatial_attributes = normalized_spatial_attributes.T  # (N, num_attrs)

    # 4. Physical channel properties — each (N,)
    flowpath_tensors = {
        # MERIT stores km; the MC engine requires metres — multiply by 1000
        "length": fill_nans(
            torch.tensor(flowpath_attr["lengthkm"].values, dtype=torch.float32),
            row_means=self.phys_means[0],
        ) * 1000,
        "slope": fill_nans(
            torch.tensor(flowpath_attr["slope"].values, dtype=torch.float32),
            row_means=self.phys_means[1],
        ),
        # MERIT uses a fixed Muskingum x = 0.3 for all reaches
        "x": torch.full_like(
            torch.tensor(flowpath_attr["lengthkm"].values, dtype=torch.float32),
            fill_value=0.3,
        ),
        # Leopold & Maddock geometry — top_width and side_slope are derived
        # from the learned parameters, so pass empty tensors here
        "top_width": torch.empty(0),
        "side_slope": torch.empty(0),
    }

    return adjacency_matrix, spatial_attributes, normalized_spatial_attributes, flowpath_tensors

### 4.5 `_init_training` — set the four required instance attributes

The base class `collate_fn` reads `self.gage_ids` directly (in `__len__` and `__getitem__`), and `_collate_gages` reads `self.gages_adjacency` and `self.obs_reader`. None of these are set by the base class — they are a hidden contract that `_init_training` must fulfil.

**Required attributes:**

| Attribute | Type | Used by |
|---|---|---|
| `self.gage_ids` | `np.ndarray` | `__len__`, `__getitem__`, DataLoader sampling |
| `self.observations` | loaded observation dataset | `_collate_gages` → `create_hydrofabric_observations` |
| `self.gages_adjacency` | zarr group / dict | `_collate_gages` → `construct_network_matrix` |
| `self.obs_reader` | `IcechunkUSGSReader` | `_collate_gages` → `build_flow_scale_tensor` (needs `gage_dict`) |

In [ ]:
# MERIT implementation
def _init_training(self) -> None:
    if self.cfg.data_sources.gages is None or self.cfg.data_sources.gages_adjacency is None:
        raise ValueError("Training requires gages and gages_adjacency to be defined")

    # Load observations and build gage list from the store
    self.obs_reader = IcechunkUSGSReader(cfg=self.cfg)
    self.observations = self.obs_reader.read_data(dates=self.dates)
    self.gage_ids = np.array([str(_id.zfill(8)) for _id in self.obs_reader.gage_dict["STAID"]])

    # Load the per-gage adjacency structure
    self.gages_adjacency = read_zarr(Path(self.cfg.data_sources.gages_adjacency))

    # Remove headwater gages (no upstream reaches → routing is trivial)
    self.gage_ids, n_removed = filter_headwater_gages(self.gage_ids, self.gages_adjacency)
    print(f"Training mode: {len(self.gage_ids)} gauged locations ({n_removed} headwaters removed)")

### 4.6 `_init_inference` — build `self.routing_dataclass`

During inference the entire routing domain is built once in `__init__` and stored as `self.routing_dataclass`. The base class `collate_fn` reads it on every iteration, updating only the time window.

Three sub-cases must be handled in priority order:

1. **`target_catchments`** — user has specified explicit segment IDs; traverse the graph upstream.
2. **`gages` mode** — user has a gage CSV and gages adjacency; build the union of gage subgraphs.
3. **All segments** — fallback; route every segment in the domain.

In [ ]:
# MERIT implementation — abbreviated to show the three-branch structure
def _init_inference(self) -> None:
    if self.cfg.data_sources.target_catchments is not None:
        # Branch 1: explicit target catchments
        # (builds an upstream subgraph for each target using rustworkx)
        self.routing_dataclass = self._build_routing_data_target_catchments()

    elif (
        self.cfg.data_sources.gages is not None
        and self.cfg.data_sources.gages_adjacency is not None
    ):
        # Branch 2: gage mode — load observations so metrics can be computed
        self.obs_reader = IcechunkUSGSReader(cfg=self.cfg)
        self.observations = self.obs_reader.read_data(dates=self.dates)
        self.gage_ids = np.array([str(_id.zfill(8)) for _id in self.obs_reader.gage_dict["STAID"]])
        self.gages_adjacency = read_zarr(Path(self.cfg.data_sources.gages_adjacency))
        self.gage_ids, _ = filter_headwater_gages(self.gage_ids, self.gages_adjacency)
        self.routing_dataclass = self._build_routing_data_gages()

    else:
        # Branch 3: route every segment in the CONUS domain
        self.routing_dataclass = self._build_routing_data_all_catchments()

### 4.7 `_collate_gages` — build a RoutingDataclass for a training mini-batch

This is the most complex method. It is called once per mini-batch during training. The batch arrives as an array of gage IDs; you must:

1. Build the compressed subgraph (all segments upstream of every gage in the batch).
2. Call `_build_common_tensors` to get the tensors.
3. Build `outflow_idx` — the ragged list that tells the routing engine where each gage outlet is.
4. Fetch observations aligned to this batch's gages and time window.
5. Return a fully-populated `RoutingDataclass`.

The subgraph compression step is identical across datasets; `construct_network_matrix` handles it given your `gages_adjacency`.

In [ ]:
# MERIT implementation
def _collate_gages(self, batch: np.ndarray) -> RoutingDataclass:
    # Drop gages not in the adjacency structure (e.g. cross-boundary)
    valid_mask = np.isin(batch, list(self.gages_adjacency.keys()))
    batch = batch[valid_mask].tolist()

    # Build the sparse COO matrix for this batch's subgraph
    coo, _gage_idx, gage_catchment = construct_network_matrix(batch, self.gages_adjacency)

    # Identify which global indices are active in this subgraph
    edge_indices = (
        np.unique(np.concatenate([coo.row, coo.col])) if coo.nnz > 0
        else np.array([], dtype=int)
    )
    gage_indices = np.array(_gage_idx, dtype=int)
    active_indices = np.unique(np.concatenate([edge_indices, gage_indices]))

    # Remap global indices to 0-based compressed indices
    index_mapping = {orig: comp for comp, orig in enumerate(active_indices)}

    if coo.nnz > 0:
        c_rows = np.array([index_mapping[i] for i in coo.row])
        c_cols = np.array([index_mapping[i] for i in coo.col])
    else:
        c_rows = c_cols = np.array([], dtype=int)

    N = len(active_indices)
    compressed_csr = sparse.coo_matrix(
        (coo.data[:len(c_rows)], (c_rows, c_cols)), shape=(N, N)
    ).tocsr()

    compressed_ids = self.merit_ids[active_indices]
    compressed_flowpath = self.flowpath_attr.reindex(compressed_ids)

    # outflow_idx[i] = compressed column indices that drain into gage i
    outflow_idx = []
    for _idx in _gage_idx:
        mask = np.isin(coo.row, _idx)
        orig_cols = coo.col[np.where(mask)[0]]
        if len(orig_cols) > 0:
            outflow_idx.append(np.array([index_mapping[c] for c in orig_cols]))
        else:
            outflow_idx.append(np.array([index_mapping[int(_idx)]]))

    gage_compressed_indices = [index_mapping[int(i)] for i in _gage_idx]
    flow_scale = build_flow_scale_tensor(
        batch=batch,
        gage_dict=self.obs_reader.gage_dict,
        gage_compressed_indices=gage_compressed_indices,
        num_segments=N,
    )

    adjacency_matrix, spatial_attributes, normalized_spatial_attributes, flowpath_tensors = (
        self._build_common_tensors(compressed_csr, compressed_ids, compressed_flowpath)
    )

    observations = create_hydrofabric_observations(
        dates=self.dates,
        gage_ids=np.array(batch),
        observations=self.observations,
    )

    return RoutingDataclass(
        spatial_attributes=spatial_attributes,
        length=flowpath_tensors["length"],
        slope=flowpath_tensors["slope"],
        side_slope=flowpath_tensors["side_slope"],
        top_width=flowpath_tensors["top_width"],
        x=flowpath_tensors["x"],
        dates=self.dates,
        adjacency_matrix=adjacency_matrix,
        normalized_spatial_attributes=normalized_spatial_attributes,
        observations=observations,
        divide_ids=compressed_ids,
        outflow_idx=outflow_idx,
        gage_catchment=gage_catchment,
        flow_scale=flow_scale,
    )

---
## 5. The complete class

The cell below assembles all six methods into the complete MERIT implementation as it appears in `src/ddr/geodatazoo/merit.py`. Reading the final class after seeing each piece explained is the best way to understand how they fit together.

In [ ]:
import logging
import rustworkx as rx
from ddr.io.builders import _build_network_graph
from ddr.io.readers import filter_gages_by_area_threshold, filter_gages_by_da_valid

log = logging.getLogger(__name__)


class Merit(BaseGeoDataset):
    """BaseGeoDataset implementation for the MERIT-Hydro global river network.

    Attributes are stored in a multi-file NetCDF dataset indexed by COMID.
    Channel geometry (length, slope) comes from a shapefile; top_width and
    side_slope are left empty because MERIT uses Leopold & Maddock power-law
    geometry derived from the learned KAN parameters.
    """

    def __init__(self, cfg: Config) -> None:
        self.cfg = cfg
        self.dates = Dates(**self.cfg.experiment.model_dump())
        self.gage_ids: np.ndarray | None = None
        self.routing_dataclass: RoutingDataclass | None = None
        self.network_graph: rx.PyDiGraph | None = None
        self.hf_id_to_node: dict[int, int] | None = None
        self.observations: Any = None
        self.gages_adjacency: Any = None
        self.target_catchments: list[str] | None = None

        self.attribute_ds = self._load_attributes()
        self.attr_stats = set_statistics(self.cfg, self.attribute_ds)
        self.id_to_index = {comid: idx for idx, comid in enumerate(self.attribute_ds.COMID.values)}
        self.attributes_list = list(self.cfg.kan.input_var_names)

        self.means = torch.tensor(
            [self.attr_stats[attr].iloc[2] for attr in self.attributes_list],
            device=self.cfg.device, dtype=torch.float32,
        ).unsqueeze(1)
        self.stds = torch.tensor(
            [self.attr_stats[attr].iloc[3] for attr in self.attributes_list],
            device=self.cfg.device, dtype=torch.float32,
        ).unsqueeze(1)

        _flowpath_attr = gpd.read_file(self.cfg.data_sources.geospatial_fabric_gpkg).set_index("COMID")
        self.flowpath_attr = _flowpath_attr[~_flowpath_attr.index.duplicated(keep="first")]
        self.phys_means = torch.tensor(
            [naninfmean(self.flowpath_attr[attr].values) for attr in ["lengthkm", "slope"]],
            device=self.cfg.device, dtype=torch.float32,
        ).unsqueeze(1)

        self.conus_adjacency = read_zarr(Path(cfg.data_sources.conus_adjacency))
        self.merit_ids = self.conus_adjacency["order"][:]

        if cfg.mode == Mode.TRAINING:
            self._init_training()
        else:
            self._init_inference()

    # ------------------------------------------------------------------ #
    # Abstract method implementations                                      #
    # ------------------------------------------------------------------ #

    def _load_attributes(self) -> xr.Dataset:
        return xr.open_mfdataset(self.cfg.data_sources.attributes)

    def _get_attributes(
        self,
        catchment_ids: np.ndarray,
        device: str | torch.device = "cpu",
        dtype: torch.dtype = torch.float32,
    ) -> torch.Tensor:
        valid_indices, divide_idx_mask = [], []
        for i, divide_id in enumerate(catchment_ids):
            comid = int(divide_id)
            if comid in self.id_to_index:
                valid_indices.append(self.id_to_index[comid])
                divide_idx_mask.append(i)
        assert valid_indices, "No valid COMIDs found in this batch"
        output = torch.full((len(self.attributes_list), len(catchment_ids)), np.nan, device=device, dtype=dtype)
        _ds = self.attribute_ds[self.attributes_list].isel(COMID=valid_indices).compute()
        data_tensor = torch.from_numpy(_ds.to_array(dim="COMID").values).to(device=device, dtype=dtype)
        output[:, divide_idx_mask] = data_tensor
        return fill_nans(attr=output, row_means=self.means)

    def _build_common_tensors(
        self,
        csr_matrix: sparse.csr_matrix,
        catchment_ids: np.ndarray,
        flowpath_attr: gpd.GeoDataFrame,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, dict[str, torch.Tensor]]:
        adjacency_matrix = torch.sparse_csr_tensor(
            crow_indices=csr_matrix.indptr,
            col_indices=csr_matrix.indices,
            values=csr_matrix.data,
            size=csr_matrix.shape,
            device=self.cfg.device,
            dtype=torch.float32,
        )
        spatial_attributes = self._get_attributes(
            catchment_ids=catchment_ids, device=self.cfg.device, dtype=torch.float32
        )
        for r in range(spatial_attributes.shape[0]):
            nan_mask = torch.isnan(spatial_attributes[r])
            spatial_attributes[r, nan_mask] = torch.nanmean(spatial_attributes[r])
        normalized_spatial_attributes = ((spatial_attributes - self.means) / self.stds).T
        flowpath_tensors = {
            "length": fill_nans(
                torch.tensor(flowpath_attr["lengthkm"].values, dtype=torch.float32),
                row_means=self.phys_means[0],
            ) * 1000,  # km → m
            "slope": fill_nans(
                torch.tensor(flowpath_attr["slope"].values, dtype=torch.float32),
                row_means=self.phys_means[1],
            ),
            "x": torch.full_like(
                torch.tensor(flowpath_attr["lengthkm"].values, dtype=torch.float32), fill_value=0.3
            ),
            "top_width": torch.empty(0),
            "side_slope": torch.empty(0),
        }
        return adjacency_matrix, spatial_attributes, normalized_spatial_attributes, flowpath_tensors

    def _init_training(self) -> None:
        if self.cfg.data_sources.gages is None or self.cfg.data_sources.gages_adjacency is None:
            raise ValueError("Training requires gages and gages_adjacency")
        self.obs_reader = IcechunkUSGSReader(cfg=self.cfg)
        self.observations = self.obs_reader.read_data(dates=self.dates)
        self.gage_ids = np.array([str(_id.zfill(8)) for _id in self.obs_reader.gage_dict["STAID"]])
        if "DA_VALID" in self.obs_reader.gage_dict:
            self.gage_ids, n = filter_gages_by_da_valid(self.gage_ids, self.obs_reader.gage_dict)
            log.info(f"Filtered {n} gages with DA_VALID=False")
        elif self.cfg.experiment.max_area_diff_sqkm is not None:
            self.gage_ids, n = filter_gages_by_area_threshold(
                self.gage_ids, self.obs_reader.gage_dict, self.cfg.experiment.max_area_diff_sqkm
            )
            log.info(f"Filtered {n} gages exceeding area diff threshold")
        self.gages_adjacency = read_zarr(Path(self.cfg.data_sources.gages_adjacency))
        self.gage_ids, n_hw = filter_headwater_gages(self.gage_ids, self.gages_adjacency)
        log.info(f"Training: {len(self.gage_ids)} gauged locations ({n_hw} headwaters removed)")

    def _init_inference(self) -> None:
        if self.cfg.data_sources.target_catchments is not None:
            self.target_catchments = self.cfg.data_sources.target_catchments
            self.network_graph, self.hf_id_to_node, _ = _build_network_graph(self.conus_adjacency)
            self.routing_dataclass = self._build_routing_data_target_catchments()
        elif self.cfg.data_sources.gages is not None and self.cfg.data_sources.gages_adjacency is not None:
            self.obs_reader = IcechunkUSGSReader(cfg=self.cfg)
            self.observations = self.obs_reader.read_data(dates=self.dates)
            self.gage_ids = np.array([str(_id.zfill(8)) for _id in self.obs_reader.gage_dict["STAID"]])
            self.gages_adjacency = read_zarr(Path(self.cfg.data_sources.gages_adjacency))
            self.gage_ids, _ = filter_headwater_gages(self.gage_ids, self.gages_adjacency)
            self.routing_dataclass = self._build_routing_data_gages()
        else:
            self.routing_dataclass = self._build_routing_data_all_catchments()

    def _collate_gages(self, batch: np.ndarray) -> RoutingDataclass:
        valid_mask = np.isin(batch, list(self.gages_adjacency.keys()))
        batch = batch[valid_mask].tolist()
        coo, _gage_idx, gage_catchment = construct_network_matrix(batch, self.gages_adjacency)
        edge_indices = np.unique(np.concatenate([coo.row, coo.col])) if coo.nnz > 0 else np.array([], dtype=int)
        active_indices = np.unique(np.concatenate([edge_indices, np.array(_gage_idx, dtype=int)]))
        index_mapping = {orig: comp for comp, orig in enumerate(active_indices)}
        c_rows = np.array([index_mapping[i] for i in coo.row]) if coo.nnz > 0 else np.array([], dtype=int)
        c_cols = np.array([index_mapping[i] for i in coo.col]) if coo.nnz > 0 else np.array([], dtype=int)
        N = len(active_indices)
        compressed_csr = sparse.coo_matrix((coo.data[:len(c_rows)], (c_rows, c_cols)), shape=(N, N)).tocsr()
        compressed_ids = self.merit_ids[active_indices]
        compressed_fp = self.flowpath_attr.reindex(compressed_ids)
        outflow_idx = []
        for _idx in _gage_idx:
            mask = np.isin(coo.row, _idx)
            orig_cols = coo.col[np.where(mask)[0]]
            outflow_idx.append(
                np.array([index_mapping[c] for c in orig_cols]) if len(orig_cols) > 0
                else np.array([index_mapping[int(_idx)]])
            )
        flow_scale = build_flow_scale_tensor(
            batch=batch,
            gage_dict=self.obs_reader.gage_dict,
            gage_compressed_indices=[index_mapping[int(i)] for i in _gage_idx],
            num_segments=N,
        )
        adj, sa, nsa, fpt = self._build_common_tensors(compressed_csr, compressed_ids, compressed_fp)
        obs = create_hydrofabric_observations(
            dates=self.dates, gage_ids=np.array(batch), observations=self.observations
        )
        return RoutingDataclass(
            spatial_attributes=sa, length=fpt["length"], slope=fpt["slope"],
            side_slope=fpt["side_slope"], top_width=fpt["top_width"], x=fpt["x"],
            dates=self.dates, adjacency_matrix=adj, normalized_spatial_attributes=nsa,
            observations=obs, divide_ids=compressed_ids, outflow_idx=outflow_idx,
            gage_catchment=gage_catchment, flow_scale=flow_scale,
        )

---
## 6. Using your custom dataset in a training or routing run

The built-in datasets are accessed via the `GeoDataset` enum:

```python
dataset = cfg.geodataset.get_dataset_class(cfg=cfg)  # returns Merit or LynkerHydrofabric
```

For a custom subclass you instantiate it directly and pass it to the same DataLoader pattern used by the training script:

In [ ]:
import torch
from torch.utils.data import DataLoader, RandomSampler

# Instantiate your custom dataset — same signature as the built-in classes
dataset = Merit(cfg=config)  # replace Merit with your class

# The DataLoader uses the shared collate_fn from BaseGeoDataset
sampler = RandomSampler(dataset, generator=torch.Generator().manual_seed(config.seed))
dataloader = DataLoader(
    dataset=dataset,
    batch_size=config.experiment.batch_size,
    num_workers=0,
    sampler=sampler,
    collate_fn=dataset.collate_fn,
    drop_last=True,
)

# Each iteration yields a RoutingDataclass ready for the routing engine
for routing_dataclass in dataloader:
    print("Adjacency shape  :", routing_dataclass.adjacency_matrix.shape)
    print("Attributes shape :", routing_dataclass.normalized_spatial_attributes.shape)
    print("Divide IDs       :", routing_dataclass.divide_ids[:5])
    break

---
## 7. Common mistakes

| Mistake | Symptom | Fix |
|---|---|---|
| `_get_attributes` returns `(N, num_attrs)` instead of `(num_attrs, N)` | Shape mismatch in `_build_common_tensors` normalisation broadcast | Transpose before returning |
| Forgetting `.T` on `normalized_spatial_attributes` | KAN receives wrong shape; silent wrong result | Add `normalized_spatial_attributes = normalized_spatial_attributes.T` after z-scoring |
| `length` in km instead of metres | MC Courant number is 1000× too small; routing is numerically unstable | Multiply `lengthkm` by 1000 in `_build_common_tensors` |
| `_init_training` does not set `self.obs_reader` | `AttributeError` in `_collate_gages` when `build_flow_scale_tensor` accesses `gage_dict` | Set `self.obs_reader = IcechunkUSGSReader(cfg=self.cfg)` |
| `_init_inference` does not set `self.routing_dataclass` | `AssertionError: No RoutingDataclass, cannot batch` from base class `collate_fn` | Build and assign it at the end of each inference branch |
| `top_width` / `side_slope` left as `None` instead of `torch.empty(0)` | `TypeError` in the routing engine geometry branch selection | Set unused geometry tensors to `torch.empty(0)` |